# Bayesian Optimization for Black-Box Functions (Submission 3)

This notebook implements the updated Bayesian Optimization framework for **Round 3 Queries** across 8 black-box functions. 

### Rationale & Incremental Updates for Round 3:
* **Matern 5/2 Kernel:** Maintained for structural flexibility over complex landscape irregularities.
* **Enhanced GPR Exploration Stability:** Retains `normalize_y=True` to adaptively rescale multi-scale outputs smoothly while stepping up the internal GPR configuration to `n_restarts_optimizer=30` to ensure highly robust hyperparameter tuning.
* **Acquisition Scaling:** Employs Expected Improvement (EI) with an exploration constant ($\xi = 0.01$).
* **Denser Surface Optimization:** Upgraded the acquisition space optimization step to **40 random restarts** of the local L-BFGS-B search engine to avoid local trapping as history grows.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Ensure inline plotting
%matplotlib inline 

print("Environment ready. Core computation libraries loaded.")

## The Bounded Bayesian Optimizer Architecture

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes the optimizer step config.
        """
        # Function 2 handles a noisy likelihood with a robust alpha boundary
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads multi-round cumulative histories and updates the Gaussian Process Regression model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            length_scale_bounds=(0.01, 1.0), 
            nu=2.5
        )
        
        # In Round 3, n_restarts_optimizer is raised to 30 for deeper convergence exploration
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True,
            n_restarts_optimizer=30
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi=0.01):
        """
        Computes Expected Improvement (EI) surface evaluations.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self):
        """
        Maximizes the complex EI surface utilizing 40 distinct L-BFGS-B random restarts.
        """
        best_x = None
        max_ei = -1
        
        # 40 restarts accounts for the increasingly non-convex multi-modal surface
        for _ in range(40):
            x0 = np.random.uniform(0.0, 0.999999, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x),
                x0,
                bounds=[(0.0, 0.999999)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Loop Execution and Submission Generation (Functions 1-8)

In [ ]:
output_file = "submission_3_results.txt"

print("Calculating Round 3 Queries (Refinement Phase)...")
print("-" * 50)

notebook_results = {}

with open(output_file, "w") as f:
    for i in range(1, 8 + 1):
        func_dir = f"function_{i}"
        
        if not os.path.exists(func_dir):
            print(f"Warning: Directory '{func_dir}' not found. Skipping.")
            continue
            
        # Function 2 remains the designated noisy channel outlier
        optimizer = BayesianOptimizer(is_noisy=(i == 2))
        optimizer.load_and_fit(func_dir)
        
        next_point = optimizer.propose_next_point()
        formatted_point = "-".join([f"{val:.6f}" for val in next_point])
        
        result_line = f"Function {i}: {formatted_point}"
        f.write(result_line + "\n")
        
        notebook_results[f"Function {i}"] = formatted_point
        print(result_line)

print("-" * 50)
print(f"Submission 3 successfully generated and saved to: {output_file}")

### Interactive Summary Dashboard

In [ ]:
print(f"| {'Target Objective':16} | {'Proposed Query Coordinates (Round 3)':55} |")
print(f"| {'-'*16} | {'-'*55} |")
for func, points in notebook_results.items():
    print(f"| {func:16} | {points:55} |")